# SPRO Multi-Turn Jailbreak Training

**Self-contained notebook for RunPod / Colab deployment.**

## Quick Start (Pod Deployment)

1. **Run cells 1-4** to clone repo and install dependencies
2. **Set API keys** in cell 6 (`openrouter_api_key`) and cell 7 (`WANDB_API_KEY`)
3. **Select config** in cell 6 (default: `config_refusal_direction`)
4. **Run all remaining cells** to start training

## Config Options

| Config | Description | Use Case |
|--------|-------------|----------|
| `config_paper` | Paper-aligned, MSA on all tokens | Stable baseline |
| `config_paper_thinking` | Paper MSA + thinking-only | Recommended |
| `config_aggressive` | Fast iteration (5x LR) | Quick testing |
| `config_refusal_direction` | **API generation + local refusal scoring** | Best for multi-turn |

## Refusal Direction Reward (config_refusal_direction)

**Optimized architecture:**
- **API generates** target responses (fast, parallelizable)
- **Local model scores** refusal via single forward pass (no generation)

This is much faster than generating locally - the local model only does N forward passes per episode instead of N × hundreds of autoregressive steps.

**Prerequisite:** Run `notebooks/05_find_refusal_direction.ipynb` first to create the refusal probe:
```
outputs/refusal_directions/refusal_probe_{model}_layer{N}.pt
```

## Module Structure
```
src/spro/
  config.py          # SPROConfig dataclass
  local_target.py    # LocalTargetWithActivations (score_prompt_refusal)
  refusal_reward.py  # Continuity-based reward
  trainer.py         # SingleExampleSPROTrainer
  ...
```

## 1. Setup - Clone Repo & Install Dependencies

In [ ]:
import os

# Clone repo if not already present
if not os.path.exists('JB_mech'):
    !git clone https://github.com/ChuloIva/JB_mech.git
    print("Repo cloned!")
else:
    # Pull latest changes
    !cd JB_mech && git pull
    print("Repo already exists, pulled latest.")

In [ ]:
%%capture
# Install dependencies
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    import torch; import re; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets>=4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install sentence-transformers openai
!pip install liger-kernel vllm
!pip install nest_asyncio tenacity httpx aiohttp

In [ ]:
# Add src to path for imports
import sys
sys.path.insert(0, 'JB_mech/src')

print("Installation complete!")

## 2. Configuration

In [ ]:
from spro import SPROConfig
from pathlib import Path

# =============================================================================
# PATHS (auto-detected based on environment)
# =============================================================================
# Detect if we're in cloned repo or local development
REPO_ROOT = Path("JB_mech") if Path("JB_mech").exists() else Path(".")
REFUSAL_DIR = REPO_ROOT / "outputs" / "refusal_directions"

# Auto-find refusal probe for target model
def find_refusal_probe(model_name: str, probe_dir: Path = REFUSAL_DIR) -> str:
    """Find refusal probe file for a given model."""
    if not probe_dir.exists():
        return ""
    
    # Convert model name to filename pattern
    model_slug = model_name.replace("/", "_")
    
    # Look for probe file (minimal version with best layer)
    probe_files = list(probe_dir.glob(f"refusal_probe_{model_slug}_layer*.pt"))
    if probe_files:
        # Return the first match (there should typically be one)
        return str(probe_files[0])
    
    # Fallback to full direction file
    full_files = list(probe_dir.glob(f"refusal_direction_{model_slug}.pt"))
    if full_files:
        return str(full_files[0])
    
    return ""

# =============================================================================
# CONFIG OPTION 1: Paper-Aligned (Stable, Whole Rollout)
# Based on arXiv:2507.01551 - MSA on all tokens
# =============================================================================
config_paper = SPROConfig(
    # Model
    model_name="Koalacrown/qwen3-4b-multiturn-sft-16bit",
    lora_rank=32,
    
    # Training - Paper settings
    learning_rate=1e-6,           # Paper: 1e-6 with cosine decay
    kl_coef=0.0,                  # Paper: 0 (no KL penalty)
    entropy_coef=0.001,           # Paper: 0.001 (maintains exploration)
    batch_ppo_update=True,        # Paper: batched updates
    group_size=8,
    max_attempts=200,
    
    # SPRO advantages - Paper settings (whole rollout)
    use_paper_msa=True,           # Use exact paper MSA formula
    thinking_only_msa=False,      # MSA on ALL tokens
    use_three_component_msa=False,
    msa_scale=1.0,
    advantage_clip=5.0,
    normalize_advantages=False,
    beta=1.0,
    
    # Disable other MSA variants
    use_gae_advantages=False,
    use_query_aware_msa=False,

    # Rewards
    outcome_weight=0.85,
    divergence_weight=0.15,
    
    # API
    openrouter_api_key="",
    target_model="google/gemma-2-9b-it",
    judge_model="google/gemma-3-27b-it",
    
    # Logging & Checkpointing
    log_rollouts=True,
    rollout_log_dir="./rollout_logs",
    checkpoint_dir="./spro_checkpoints",
)

# =============================================================================
# CONFIG OPTION 2: Paper MSA + Thinking-Only (Recommended)
# Paper's simple MSA formula, but focused on reasoning section
# =============================================================================
config_paper_thinking = SPROConfig(
    # Model
    model_name="Koalacrown/qwen3-4b-multiturn-sft-16bit",
    lora_rank=32,
    
    # Training - Paper settings
    learning_rate=1e-6,           # Paper: 1e-6
    kl_coef=0.0,                  # Paper: 0
    entropy_coef=0.001,           # Paper: 0.001
    batch_ppo_update=True,        # Paper: batched updates
    group_size=8,
    max_attempts=200,
    
    # SPRO advantages - Paper MSA on thinking section only
    use_paper_msa=True,           # Paper's simple baseline subtraction
    thinking_only_msa=True,       # MSA only on reasoning/thinking tokens
    use_three_component_msa=False,
    msa_scale=1.0,
    advantage_clip=5.0,
    normalize_advantages=False,
    beta=1.0,
    
    # Disable other MSA variants
    use_gae_advantages=False,
    use_query_aware_msa=False,

    # Rewards
    outcome_weight=0.85,
    divergence_weight=0.15,
    
    # API
    openrouter_api_key="",
    target_model="google/gemma-2-9b-it",
    judge_model="google/gemma-3-27b-it",
    
    # Logging & Checkpointing
    log_rollouts=True,
    rollout_log_dir="./rollout_logs",
    checkpoint_dir="./spro_checkpoints",
)

# =============================================================================
# CONFIG OPTION 3: Aggressive (Fast Learning)
# For quick iteration/testing - may converge in ~20 steps but less stable
# =============================================================================
config_aggressive = SPROConfig(
    # Model
    model_name="Koalacrown/qwen3-4b-multiturn-sft-16bit",
    lora_rank=32,
    
    # Training - Aggressive settings
    learning_rate=5e-6,           # 5x paper LR (faster but riskier)
    kl_coef=0.01,                 # Small KL to prevent collapse
    entropy_coef=0.002,           # 2x paper entropy (more exploration)
    batch_ppo_update=True,        # Still use batched updates
    group_size=8,
    max_attempts=50,              # Fewer attempts per intent
    
    # SPRO advantages - Paper MSA on thinking only
    use_paper_msa=True,
    thinking_only_msa=True,       # Focus on reasoning section
    use_three_component_msa=False,
    msa_scale=1.0,
    advantage_clip=3.0,           # Tighter clipping for stability
    normalize_advantages=False,
    beta=1.0,
    
    # Disable other MSA variants
    use_gae_advantages=False,
    use_query_aware_msa=False,

    # Rewards
    outcome_weight=0.85,
    divergence_weight=0.15,
    
    # API
    openrouter_api_key="",
    target_model="google/gemma-2-9b-it",
    judge_model="google/gemma-3-27b-it",
    
    # Logging & Checkpointing
    log_rollouts=True,
    rollout_log_dir="./rollout_logs",
    checkpoint_dir="./spro_checkpoints",
)

# =============================================================================
# CONFIG OPTION 4: Refusal Direction Reward (API + Local Scoring)
# - API generates target responses (fast, parallelizable)
# - Local model scores refusal via forward pass (no generation)
# =============================================================================
# Target model (used for both API generation and local refusal scoring)
LOCAL_TARGET_MODEL = "google/gemma-2-9b-it"

config_refusal_direction = SPROConfig(
    # Model
    model_name="Koalacrown/qwen3-14b-multiturn-sft-16bit",
    lora_rank=32,
    
    # Training
    learning_rate=5e-6,
    kl_coef=0.0,
    entropy_coef=0.002,
    batch_ppo_update=True,
    group_size=16,
    max_attempts=100,
    
    # SPRO advantages
    use_paper_msa=True,
    thinking_only_msa=False,
    msa_scale=1.0,
    advantage_clip=5.0,
    beta=1.0,
    
    # Standard rewards (IDA still used for final scoring)
    outcome_weight=1,
    divergence_weight=0,
    
    # API - target model for generation, judge for scoring
    openrouter_api_key="",
    target_model=LOCAL_TARGET_MODEL,  # API uses same model as local
    judge_model="google/gemma-3-27b-it",
    
    # ==========================================================================
    # REFUSAL DIRECTION REWARD SETTINGS
    # ==========================================================================
    use_refusal_direction=True,   # Enable refusal direction reward
    
    # Local target model (for activation capture / refusal scoring only)
    local_target_model=LOCAL_TARGET_MODEL,
    local_target_load_in_4bit=True,
    local_target_device="auto",
    
    # Refusal probe (auto-detected from outputs/refusal_directions/)
    refusal_direction_path=find_refusal_probe(LOCAL_TARGET_MODEL),
    refusal_probe_layers=[15, 20, 25, 30],  # Layers to capture activations from
    
    # Continuity reward weights
    continuity_weight=0.35,       # Weight on unbroken acceptance chain
    smoothness_weight=0.25,       # Weight on gradual escalation
    min_score_weight=0.25,        # Weight on weakest-link score
    final_score_weight=0.15,      # Weight on final state
    
    # Reward combination (refusal vs IDA)
    refusal_reward_weight=0.4,    # Initial weight for refusal reward
    anneal_refusal_reward=False,  # Decrease refusal weight over training
    refusal_anneal_start=0.5,     # Starting refusal weight
    refusal_anneal_end=0.1,       # Final refusal weight
    refusal_anneal_steps=100,     # Steps to anneal over
    
    # Early stopping (abort dead-end trajectories)
    enable_early_stopping=True,
    early_stop_threshold=-0.3,
    recovery_window=2,
    
    # Logging & Checkpointing
    log_rollouts=True,
    rollout_log_dir="./rollout_logs",
    checkpoint_dir="./spro_checkpoints",
)

# =============================================================================
# SELECT CONFIG
# =============================================================================
# config = config_paper              # Paper-aligned, whole rollout MSA
# config = config_paper_thinking     # Paper MSA + thinking-only (recommended)
# config = config_aggressive         # Fast iteration (default for testing)
config = config_refusal_direction    # Refusal direction reward (API + local scoring)

print(f"Config loaded:")
print(f"  Type: {'REFUSAL_DIR' if config.use_refusal_direction else 'PAPER' if config.learning_rate == 1e-6 else 'AGGRESSIVE'}")
print(f"  Learning rate: {config.learning_rate}")
print(f"  KL coef: {config.kl_coef}")
print(f"  Entropy coef: {config.entropy_coef}")
print(f"  Batch PPO: {config.batch_ppo_update}")
print(f"  Paper MSA: {config.use_paper_msa}")
print(f"  Thinking-only MSA: {config.thinking_only_msa}")
print(f"  Log rollouts: {config.log_rollouts} -> {config.rollout_log_dir}")
print(f"  Group size: {config.group_size}")
if config.use_refusal_direction:
    print(f"  Refusal direction: ENABLED")
    print(f"    Target model (API): {config.target_model}")
    print(f"    Local scoring model: {config.local_target_model}")
    print(f"    Probe path: {config.refusal_direction_path or '(not found - run 05_find_refusal_direction.ipynb first)'}")
    print(f"    Probe layers: {config.refusal_probe_layers}")
    print(f"    Refusal weight: {config.refusal_reward_weight} (anneal: {config.anneal_refusal_reward})")
    print(f"    Early stopping: {config.enable_early_stopping}")

In [ ]:
import wandb
import os

# Load wandb key from .env or set directly
os.environ["WANDB_API_KEY"] = ""  # Or load from .env

# Build wandb config
wandb_config = {
    "config_type": "refusal_dir" if config.use_refusal_direction else ("paper" if config.learning_rate == 1e-6 else "aggressive"),
    "model_name": config.model_name,
    "lora_rank": config.lora_rank,
    "learning_rate": config.learning_rate,
    "kl_coef": config.kl_coef,
    "entropy_coef": config.entropy_coef,
    "batch_ppo_update": config.batch_ppo_update,
    "group_size": config.group_size,
    "use_paper_msa": config.use_paper_msa,
    "thinking_only_msa": config.thinking_only_msa,
    "msa_scale": config.msa_scale,
    "advantage_clip": config.advantage_clip,
    "normalize_advantages": config.normalize_advantages,
    "target_model": config.target_model if not config.use_refusal_direction else config.local_target_model,
    "outcome_weight": config.outcome_weight,
    "divergence_weight": config.divergence_weight,
}

# Add refusal direction settings if enabled
if config.use_refusal_direction:
    wandb_config.update({
        "use_refusal_direction": True,
        "local_target_model": config.local_target_model,
        "refusal_probe_layers": config.refusal_probe_layers,
        "continuity_weight": config.continuity_weight,
        "smoothness_weight": config.smoothness_weight,
        "min_score_weight": config.min_score_weight,
        "final_score_weight": config.final_score_weight,
        "refusal_reward_weight": config.refusal_reward_weight,
        "anneal_refusal_reward": config.anneal_refusal_reward,
        "enable_early_stopping": config.enable_early_stopping,
    })

wandb.init(
    project="spro-jailbreak",
    config=wandb_config,
)
print("Wandb initialized!")

## 3. HuggingFace Login & Setup

In [ ]:
# from huggingface_hub import login
# login()

In [ ]:
# import os

# # Mount Google Drive (Colab) or create local checkpoint dir
# try:
#     from google.colab import drive
#     drive.mount('/content/drive')
#     config.checkpoint_dir = "/content/drive/MyDrive/mt_rl_checkpoints/spro"
#     print("Google Drive mounted.")
# except ImportError:
#     print("Not in Colab, using local checkpoint directory.")

# os.makedirs(config.checkpoint_dir, exist_ok=True)
# os.makedirs(f"{config.checkpoint_dir}/rollouts", exist_ok=True)
# print(f"Checkpoints: {config.checkpoint_dir}")

## 4. Load Models

**Multi-GPU Setup:**
- GPU 0: Policy model (training, needs gradients)
- GPU 1: Reference model + Local target (inference only)

In [ ]:
import torch

# =============================================================================
# GPU CONFIGURATION
# =============================================================================
n_gpus = torch.cuda.device_count()
print(f"Available GPUs: {n_gpus}")
for i in range(n_gpus):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name} ({mem:.1f} GB)")

# Device assignments (adjust based on your setup)
if n_gpus >= 2:
    POLICY_DEVICE = "cuda:0"      # Policy model (training)
    REF_DEVICE = "cuda:1"         # Reference model (inference)
    TARGET_DEVICE = "cuda:1"      # Local target (inference)
    print(f"\nMulti-GPU mode: Policy->GPU0, Ref+Target->GPU1")
else:
    POLICY_DEVICE = "auto"
    REF_DEVICE = "auto"
    TARGET_DEVICE = "auto"
    print(f"\nSingle-GPU mode: all models on same device")

# =============================================================================
# LOAD POLICY MODEL (GPU 0)
# =============================================================================
from unsloth import FastLanguageModel

print(f"\nLoading policy model on {POLICY_DEVICE}...")

# For multi-GPU, we need to set CUDA_VISIBLE_DEVICES or use device_map
if POLICY_DEVICE == "cuda:0" and n_gpus >= 2:
    # Load on specific GPU
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=config.model_name,
        max_seq_length=config.max_seq_length,
        load_in_4bit=False,
        fast_inference=True,
        max_lora_rank=config.lora_rank,
        gpu_memory_utilization=config.gpu_memory_utilization,
        device_map={"": 0},  # Force to GPU 0
    )
else:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=config.model_name,
        max_seq_length=config.max_seq_length,
        load_in_4bit=False,
        fast_inference=True,
        max_lora_rank=config.lora_rank,
        gpu_memory_utilization=config.gpu_memory_utilization,
    )

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=config.lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=config.lora_rank * 2,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print(f"Policy model loaded: {config.model_name}")

In [ ]:
# =============================================================================
# LOAD REFERENCE MODEL (GPU 1 if available)
# =============================================================================
print(f"\nLoading reference model on {REF_DEVICE}...")

if REF_DEVICE == "cuda:1" and n_gpus >= 2:
    ref_model, _ = FastLanguageModel.from_pretrained(
        model_name=config.model_name,
        max_seq_length=config.max_seq_length,
        load_in_4bit=False,
        fast_inference=False,
        device_map={"": 1},  # Force to GPU 1
    )
else:
    ref_model, _ = FastLanguageModel.from_pretrained(
        model_name=config.model_name,
        max_seq_length=config.max_seq_length,
        load_in_4bit=False,
        fast_inference=False,
    )

for param in ref_model.parameters():
    param.requires_grad = False

print(f"Reference model loaded and frozen.")

In [ ]:
# =============================================================================
# LOAD LOCAL TARGET MODEL (GPU 1 if available)
# =============================================================================
# PREREQUISITE: Run notebooks/05_find_refusal_direction.ipynb first to create:
#   outputs/refusal_directions/refusal_probe_{model_slug}_layer{N}.pt

local_target = None

if config.use_refusal_direction:
    from spro import LocalTargetWithActivations
    
    print("="*60)
    print("LOADING LOCAL TARGET FOR REFUSAL DIRECTION REWARD")
    print("="*60)
    
    # Check if probe exists
    if not config.refusal_direction_path:
        print(f"\n[!] No refusal probe found for: {config.local_target_model}")
        print(f"    Searched in: {REFUSAL_DIR}")
        print(f"\n    To create the probe, run:")
        print(f"    1. Open notebooks/05_find_refusal_direction.ipynb")
        print(f"    2. Set config.model_name = '{config.local_target_model}'")
        print(f"    3. Run all cells")
        print(f"    4. Re-run this notebook")
        print(f"\n    Or manually set: config.refusal_direction_path = 'path/to/probe.pt'")
        raise FileNotFoundError(f"Refusal probe not found. Run 05_find_refusal_direction.ipynb first.")
    
    # Determine device for local target
    if TARGET_DEVICE == "cuda:1" and n_gpus >= 2:
        target_device_map = {"": 1}
    else:
        target_device_map = "auto"
    
    print(f"Target model: {config.local_target_model}")
    print(f"Device: {TARGET_DEVICE}")
    print(f"Probe file: {config.refusal_direction_path}")
    print(f"Capture layers: {config.refusal_probe_layers}")
    print()
    
    local_target = LocalTargetWithActivations(
        model_name=config.local_target_model,
        layers=config.refusal_probe_layers,
        load_in_4bit=config.local_target_load_in_4bit,
        device_map=target_device_map,
    )
    
    # Load refusal direction
    local_target.load_refusal_direction(config.refusal_direction_path)
    
    print(f"\nLocal target loaded successfully!")
    print(f"  Model: {config.local_target_model}")
    if local_target.refusal_layer is not None:
        print(f"  Refusal probe: layer {local_target.refusal_layer}")
    if local_target.refusal_separation:
        print(f"  Separation: {local_target.refusal_separation:.4f}")
else:
    print("Refusal direction disabled, using API-based target.")

# Print GPU memory usage
if n_gpus >= 1:
    print(f"\nGPU Memory Usage:")
    for i in range(n_gpus):
        allocated = torch.cuda.memory_allocated(i) / 1e9
        reserved = torch.cuda.memory_reserved(i) / 1e9
        print(f"  GPU {i}: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved")

## 5. Setup Chat Template

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen3-thinking")

# Fix template to ensure <think> tag
test_msgs = [{"role": "user", "content": "test"}]
test_output = tokenizer.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True)

if "<think>" not in test_output:
    print("[!] Applying <think> tag fix...")
    tokenizer.chat_template = '''{% for message in messages %}{% if message['role'] == 'system' %}<|im_start|>system
{{ message['content'] }}<|im_end|>
{% elif message['role'] == 'user' %}<|im_start|>user
{{ message['content'] }}<|im_end|>
{% elif message['role'] == 'assistant' %}<|im_start|>assistant
{{ message['content'] }}<|im_end|>
{% endif %}{% endfor %}{% if add_generation_prompt %}<|im_start|>assistant
<think>
{% endif %}'''

test_output = tokenizer.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True)
print(f"Template contains <think>: {'<think>' in test_output}")

## 6. Load Dataset

In [ ]:
from datasets import load_dataset

# Load OBLITERATUS dataset
obliteratus = load_dataset("Koalacrown/obliteratus-jailbreak-prompts", split="train")
print(f"Loaded {len(obliteratus)} total examples")

# Filter by tier
tier1_data = obliteratus.filter(lambda x: x["tier"] == "Tier 1: Standard")
tier2_data = obliteratus.filter(lambda x: x["tier"] == "Tier 2: Elevated")
tier3_data = obliteratus.filter(lambda x: x["tier"] == "Tier 3: Critical")

print(f"Tier 1 (easy): {len(tier1_data)}")
print(f"Tier 2 (medium): {len(tier2_data)}")
print(f"Tier 3 (hard): {len(tier3_data)}")

# Train on Tier 2
training_intents = [ex["harmful"] for ex in tier2_data]
print(f"\nTraining on {len(training_intents)} Tier 2 intents")

## 7. Initialize Trainer

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from spro import IntentTracker, SingleExampleSPROTrainer

# Initialize components
intent_tracker = IntentTracker()

# Initialize trainer (pass local_target if pre-loaded)
trainer = SingleExampleSPROTrainer(
    policy_model=model,
    ref_model=ref_model,
    tokenizer=tokenizer,
    intent_tracker=intent_tracker,
    config=config,
    local_target=local_target,  # Pass pre-loaded target (or None to auto-load)
)

print("Trainer initialized.")
print(f"Training on {len(training_intents)} intents")
print(f"Group size: {config.group_size}")
if config.use_refusal_direction:
    print(f"Refusal direction: ENABLED (local target)")
    print(f"  Continuity weight: {config.continuity_weight}")
    print(f"  Early stopping: {config.enable_early_stopping}")

## 8. Test API Connection

In [ ]:
import asyncio

if config.openrouter_api_key:
    result = asyncio.get_event_loop().run_until_complete(
        trainer.api_client.test_connection()
    )
    print(f"API test: {result[:50]}...")
else:
    print("[!] Set config.openrouter_api_key to test API")

## 9. Training

In [ ]:
from tqdm.auto import tqdm

async def train_all_intents():
    """Train on all intents sequentially."""
    results = []
    
    for i, intent in enumerate(tqdm(training_intents, desc="Training intents")):
        print(f"\n{'='*60}")
        print(f"Intent {i+1}/{len(training_intents)}")
        print(f"{'='*60}")
        
        result = await trainer.train_on_intent(intent)
        results.append(result)
        trainer.training_stats.append(result)
        
        # Build wandb log dict
        log_dict = {
            "intent_idx": i,
            "success": result["success"],
            "attempts": result["attempts"],
            "best_score": result["best_score"],
            "policy_loss": result.get("policy_loss", 0),
            "success_rate": sum(1 for r in results if r["success"]) / len(results),
            "avg_attempts": sum(r["attempts"] for r in results) / len(results),
            "training_step": trainer.training_step,
        }
        
        # Add refusal-specific metrics if enabled
        if config.use_refusal_direction:
            log_dict["refusal_weight"] = config.get_refusal_weight(trainer.training_step)
        
        wandb.log(log_dict)
        
        # Save checkpoint periodically
        # if (i + 1) % config.save_every_n_intents == 0:
        #     trainer.save_checkpoint(f"{config.checkpoint_dir}/checkpoint_{i+1}")
    
    wandb.finish()
    return results

# Run training (uncomment)
# results = asyncio.get_event_loop().run_until_complete(train_all_intents())

In [ ]:
# Quick test with a few intents
async def train_quick_test(n_intents: int = 3):
    """Quick test training on a few intents."""
    results = []
    for i, intent in enumerate(training_intents[:n_intents]):
        print(f"\nIntent {i+1}/{n_intents}")
        result = await trainer.train_on_intent(intent)
        results.append(result)
        trainer.training_stats.append(result)
        
        # Log to wandb
        log_dict = {
            "intent_idx": i,
            "success": result["success"],
            "attempts": result["attempts"],
            "best_score": result["best_score"],
            "policy_loss": result.get("policy_loss", 0),
            "success_rate": sum(1 for r in results if r["success"]) / len(results),
        }
        if config.use_refusal_direction:
            log_dict["refusal_weight"] = config.get_refusal_weight(trainer.training_step)
        wandb.log(log_dict)
    return results

# Run quick test (uncomment)
# results = asyncio.get_event_loop().run_until_complete(train_quick_test(1))

## 10. Analysis

In [ ]:
def analyze_results(trainer):
    """Analyze training results."""
    stats = trainer.training_stats
    
    if not stats:
        print("No training stats yet.")
        return
    
    successes = sum(1 for s in stats if s["success"])
    total = len(stats)
    
    print(f"\n{'='*60}")
    print("TRAINING SUMMARY")
    print(f"{'='*60}")
    print(f"Intents trained: {total}")
    print(f"Successes: {successes} ({100*successes/total:.1f}%)")
    
    attempts = [s["attempts"] for s in stats]
    print(f"\nAttempts per intent: min={min(attempts)}, max={max(attempts)}, mean={sum(attempts)/len(attempts):.1f}")
    
    scores = [s["best_score"] for s in stats]
    print(f"\nBest score distribution:")
    for score in [1, 2, 3, 4]:
        count = scores.count(score)
        print(f"  Score {score}: {count}")

# analyze_results(trainer)

## 11. Save Checkpoint

In [ ]:
# trainer.save_checkpoint(f"{config.checkpoint_dir}/final")

In [ ]:
# Push to HuggingFace
# HF_USERNAME = "Koalacrown"
# model.push_to_hub(f"{HF_USERNAME}/qwen3-4b-spro-multiturn-lora")
# tokenizer.push_to_hub(f"{HF_USERNAME}/qwen3-4b-spro-multiturn-lora")